In [2]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_curve, roc_auc_score
from scipy.stats import ks_2samp
#import cupy as cp
import xgb_utils
import numpy as np
import pandas as pd
import math
import pickle as pickle
import os
import shutil
import uproot
import sys
import optuna
import optuna.visualization as vis
import matplotlib.pyplot as plt
import mplhep as hep
import plotly.io as pio
from plotly.io import show

pio.renderers.default = "notebook"
%matplotlib inline

/home/esledge/micromamba/envs/xgboost_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
!nvidia-smi

Tue Jul 21 15:41:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.71.05              Driver Version: 595.71.05      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          On  |   00000000:07:00.0 Off |                   On |
| N/A   47C    P0            151W /  300W |                  N/A   |     N/A      Default |
|                                         |                        |              Enabled |
+-----------------------------------------+-----

In [ ]:
channel_US, era = "ggH", "Combined"
background_subset = "Full"
signal_subset = "NottH"
SIGNAL_REGION = (115, 135)
plot_dir = "BDTplots/"

variables = xgb_utils.get_variables(channel_US, True)
print("number of variables", len(variables) - 3)
for i, var in enumerate(variables[:-3]):
    print(var[0])

In [ ]:
test_name = f"{channel_US}_{era}_B{background_subset}_S{signal_subset}"
plot_path = f"../../plots/xgboost/{channel_US}/B{background_subset}_S{signal_subset}/"
data_directory = f"../../root_io/skim/{channel_US}/"
signal_file_name = f"{data_directory}signal_{era}_skim_{signal_subset}.root"
bkg_file_name = f"{data_directory}background_{era}_skim_{background_subset}.root"

print("[INFO]: Creating data frames")
with uproot.open(signal_file_name) as file:
    df_signal = pd.DataFrame(
        file["tree_output"].arrays([var[0] for var in variables], library="np"),
    )
with uproot.open(bkg_file_name) as file:
    df_bkg = pd.DataFrame(
        file["tree_output"].arrays([var[0] for var in variables], library="np"),
    )
print("[INFO]: Data frames created successfully")

In [ ]:
##DOES THIS MAKE SENSE TO DO??
mass_var_name = "diMuon_bsConstrainedMass"
bkg_events = df_bkg["weight"][
    (df_bkg[mass_var_name] > SIGNAL_REGION[0])
    & (df_bkg[mass_var_name] < SIGNAL_REGION[1])
].sum()
signal_events = df_signal["weight"][
    (df_signal[mass_var_name] > SIGNAL_REGION[0])
    & (df_signal[mass_var_name] < SIGNAL_REGION[1])
].sum()

print("Bkg events: " + str(bkg_events))
print("Sig events: " + str(signal_events))

In [ ]:
xgb_utils.draw_correlation_matrix(signal_file_name, bkg_file_name, variables[:-2], plot_path, test_name)

In [ ]:
df_signal

In [ ]:
df_bkg

In [ ]:
signal_weight_sum = df_signal["weight"].sum()
bkg_weight_sum = df_bkg["weight"].sum()
print("Sig weight initial: " + str(signal_weight_sum))
print("Bkg weight initial: " + str(bkg_weight_sum))

df_signal["weight"] = df_signal["weight"] * (bkg_weight_sum / signal_weight_sum)

signal_weight_sum_post = df_signal["weight"].sum()
bkg_weight_sum_post = df_bkg["weight"].sum()
print("Sig weight post: " + str(signal_weight_sum_post))
print("Bkg weight post: " + str(bkg_weight_sum_post))

In [ ]:
print("Sig weight initial: " + str(signal_weight_sum))
print("Bkg weight initial: " + str(bkg_weight_sum))

In [ ]:
s_over_sqrt_b = signal_events / math.sqrt(bkg_events)
print(
    f"[INFO]: S = {signal_events:.3f}; B = {bkg_events:.3f}; S/sqrt(B) = {s_over_sqrt_b:.3f}"
)
print(f"[INFO]: Signal sample size: {len(df_signal.values)}")
print(f"[INFO]: Bkg sample size: {len(df_bkg.values)}")

In [ ]:
x = np.concatenate([df_bkg.values, df_signal.values])
y = np.concatenate([np.zeros(len(df_bkg)), np.ones(len(df_signal))])

In [ ]:
x

In [ ]:
y

In [ ]:
SEED = 7
TEST_SIZE = 0.4
SAMPLE_SIZE = 1.0
x_train_p, x_test_p, y_train, y_test = train_test_split(
    x,
    y,
    train_size=SAMPLE_SIZE * (1 - TEST_SIZE),
    test_size=SAMPLE_SIZE * TEST_SIZE,
    random_state=SEED,
)

In [ ]:
x_train = x_train_p[:, :-3]
x_test = x_test_p[:, :-3]
sample_weights_train = x_train_p[:, -1]
sample_weights_test = x_test_p[:, -1]

In [ ]:
htrain, bins = np.histogram(sample_weights_train)
htest, binstest = np.histogram(sample_weights_test, bins=bins)
fig, ax = plt.subplots()
hep.histplot(htrain, bins, label="train")
hep.histplot(htest, bins, label='test')
ax.set_yscale('log')
ax.set_xlabel("Sample Weight")
ax.set_ylabel("Count")
ax.legend()
plt.savefig(plot_dir + "Weights.png")
plt.savefig(plot_dir + "Weights.pdf")
plt.show()

In [ ]:
print("-inf position (by index):", np.isneginf(x_train)[np.isneginf(x_train)])
print("inf position (by index):", np.isinf(x_train)[np.isinf(x_train)])
print("nan position (by index):", np.isnan(x_train)[np.isnan(x_train)])
x_train[np.isneginf(x_train)] = np.nan
x_train[np.isinf(x_train)] = np.nan
x_train[np.isnan(x_train)] = 0.0

In [ ]:
mean_map = {}
std_map = {}
#for i, var in enumerate(variables[:-3]):
for i, var in enumerate(variables[:-3]):
    print("standarizing : ", var[0])
    print("Iterator : ", i)
    null_value = 0
    if var[0] in ["delta_phi_diJet", "pt_balance"]:
        null_value = -1

    mask_train = x_train[:, i] != null_value
    mask_test = x_test[:, i] != null_value
    mean_map[var[0]] = x_train[mask_train, i].mean()
    std_map[var[0]] = x_train[mask_train, i].std()
    print("[Info]: Train Sample Mean: ", mean_map[var[0]])
    print("[Info]: Train Sample std: ", std_map[var[0]])
    
    h_pre_train, binspre = np.histogram(x_train[mask_train, i])
    h_pre_test, bins = np.histogram(x_test[mask_test, i], bins=binspre)
    fig, ax = plt.subplots()
    hep.histplot(h_pre_train, binspre, label="train")
    hep.histplot(h_pre_test, binspre, label="test")
    #ax.set_yscale('log')
    ax.set_xlabel(var[0])
    ax.set_ylabel("Count")
    ax.legend()
    plt.title("Pre standardization")
    plt.savefig(plot_dir + var[0] + "_prestd")
    #plt.show()
    plt.close()
    
    x_train[mask_train, i] = (x_train[mask_train, i] - mean_map[var[0]]) / std_map[
        var[0]
    ]
    x_test[mask_test, i] = (x_test[mask_test, i] - mean_map[var[0]]) / std_map[
        var[0]
    ]
    
    h_post_train, binspost = np.histogram(x_train[mask_train, i])
    h_post_test, bins = np.histogram(x_test[mask_test, i], bins=binspost)
    fig, ax = plt.subplots()
    hep.histplot(h_post_train, binspost, label="train")
    hep.histplot(h_post_test, binspost, label="test")
    #ax.set_yscale('log')
    ax.set_xlabel(var[0])
    ax.set_ylabel("Count")
    ax.legend()
    plt.title("Post standardization")
    plt.savefig(plot_dir + var[0] + "_poststd")
    #plt.show()
    plt.close()
    
    #break

In [ ]:
w_train = np.absolute(sample_weights_train) / np.array(x_train_p[:, -2])
w_test = np.absolute(sample_weights_test) / np.array(x_test_p[:, -2])

In [ ]:
np.sum(w_train)*0.0075

In [ ]:
y_train_opt = y_train.copy()
y_train_nominal = y_train.copy()
y_train_run2 = y_train.copy()

y_test_opt = y_test.copy()
y_test_nominal = y_test.copy()
y_test_run2 = y_test.copy()

In [ ]:
opted_params = {'max_depth': 5, 'learning_rate': 0.17065960988185175, 'min_child_weight': 1.258974431208238,
                            'subsample': 0.9799252299722975, 'colsample_bytree': 0.9857090292326347, 
                            'reg_lambda': 0.17418443431820596, 'reg_alpha': 0.03970459451078197}

opt_model = xgb.XGBClassifier(
            **opted_params,
            n_estimators=1000,
            verbosity=2,
            n_jobs=4,
            eval_metric=["logloss", "auc"],
            early_stopping_rounds=50,
            device='cuda',
        )

nominal_model = xgb.XGBClassifier(
            max_depth=3,
            learning_rate=0.1,
            n_estimators=400,
            verbosity=2,
            n_jobs=4,
            eval_metric=["logloss", "auc"],
            device='cuda',
        )

run2_model = xgb.XGBClassifier(
            max_depth = 4,
            n_estimators=1000,
            min_child_weight = 500000,
            learning_rate = 0.1,
            subsample = 0.5,
            verbosity=2,
            n_jobs=4,
            eval_metric=["logloss", "auc"],
            device='cuda',
        )

opt_model.fit(
            x_train,
            y_train_opt,
            sample_weight=w_train,
            eval_set=[(x_train, y_train_opt), (x_test, y_test_opt)],
            sample_weight_eval_set=[w_train, w_test],
        )

nominal_model.fit(
            x_train,
            y_train_nominal,
            sample_weight=w_train,
            eval_set=[(x_train, y_train_nominal), (x_test, y_test_nominal)],
            sample_weight_eval_set=[w_train, w_test],
        )

run2_model.fit(
            x_train,
            y_train_run2,
            sample_weight=w_train,
            eval_set=[(x_train, y_train_run2), (x_test, y_test_run2)],
            sample_weight_eval_set=[w_train, w_test],
        )

In [ ]:
y_pred_opt_test = opt_model.predict_proba(x_test)[:, 1]
y_pred_opt_train = opt_model.predict_proba(x_train)[:, 1]
x_total = np.concatenate((x_test,x_train))
y_opt = np.concatenate((y_pred_opt_test,y_pred_opt_train))

y_pred_nominal_test = nominal_model.predict_proba(x_test)[:, 1]
y_pred_nominal_train = nominal_model.predict_proba(x_train)[:, 1]
y_nominal = np.concatenate((y_pred_nominal_test,y_pred_nominal_train))

y_pred_run2_test = run2_model.predict_proba(x_test)[:, 1]
y_pred_run2_train = run2_model.predict_proba(x_train)[:, 1]
y_run2 = np.concatenate((y_pred_nominal_test,y_pred_nominal_train))

In [ ]:
len(x_test_p[:, -3][y_test == 1])

In [ ]:
diMuon_mass_BSC = np.concatenate((x_test_p[:, -3][y_test == 0],x_train_p[:, -3][y_train == 0]))
print(diMuon_mass_BSC)
total_weights = np.concatenate((x_test_p[:, -1][y_test == 0],x_train_p[:, -1][y_train == 0]))
counts, bins = np.histogram(diMuon_mass_BSC, bins=100, weights=total_weights)

fig, ax = plt.subplots()

hep.histplot(counts, bins=bins, ax=ax, label="hi")
hep.cms.label("Preliminary", data=False, com=13) 
ax.set_xlabel('$M_{\mu\mu}$')
ax.set_ylabel('Events')
ax.legend()

In [ ]:
print(y_pred_opt_test)
y_pred_run2_test_bkg = run2_model.predict_proba(x_test[y_test == 0])[:, 1]
y_pred_run2_test_sig = run2_model.predict_proba(x_test[y_test == 1])[:, 1]
counts_bkg_test, bins = np.histogram(y_pred_run2_test_bkg, bins=100, weights=w_test[y_train == 0])
counts_sig_test, bins = np.histogram(y_pred_run2_test_sig, bins=100, weights=w_test[y_train == 1])

y_pred_run2_train_bkg = run2_model.predict_proba(x_train[y_train == 0])[:, 1]
y_pred_run2_train_sig = run2_model.predict_proba(x_train[y_train == 1])[:, 1]
counts_bkg_train, bins = np.histogram(y_pred_run2_train_bkg, bins=100, weights=w_train[y_train == 0])
counts_sig_train, bins = np.histogram(y_pred_run2_train_sig, bins=100, weights=w_train[y_train == 1])

fig, ax = plt.subplots()

hep.histplot(counts_bkg_test, bins=bins, ax=ax, label="bkg test")
hep.histplot(counts_sig_test, bins=bins, ax=ax, label="sig test")
hep.histplot(counts_bkg_train, bins=bins, ax=ax, label="bkg train")
hep.histplot(counts_sig_train, bins=bins, ax=ax, label="sig train")
ax.legend()

In [ ]:
bdt_divisions = 6
mass_low = 115
mass_high = 135
hep.style.use("CMS")

#fig, ax = plt.subplots()
fig, axs = plt.subplots(
    nrows=1,
    ncols=3,
    figsize=(18, 6),
    sharex=True,
)

hep.cms.label("", data=False, ax=axs[0]) 
axs[2].set_xlabel('$M_{\mu\mu}$')
axs[0].set_ylabel('Normalized Events')

bdt_score_opt = np.concatenate((y_pred_opt_test[y_test == 0],y_pred_opt_train[y_train == 0]))
bdt_score_nominal = np.concatenate((y_pred_nominal_test[y_test == 0],y_pred_nominal_train[y_train == 0]))
bdt_score_run2 = np.concatenate((y_pred_run2_test[y_test == 0],y_pred_run2_train[y_train == 0]))

for bdt_cut in np.linspace(0,1-1/bdt_divisions,bdt_divisions):
    bdt_spacing = 1/bdt_divisions
    bdt_low = bdt_cut
    bdt_high = bdt_low + bdt_spacing
    
    mask_opt = (bdt_score_opt > bdt_low) & (diMuon_mass_BSC > mass_low) & (diMuon_mass_BSC < mass_high)
    mask_nominal = (bdt_score_nominal > bdt_low) & (diMuon_mass_BSC > mass_low) & (diMuon_mass_BSC < mass_high)
    mask_run2 = (bdt_score_run2 > bdt_low) & (diMuon_mass_BSC > mass_low) & (diMuon_mass_BSC < mass_high)
    hep.style.use("CMS")

    mass_bins = np.linspace(mass_low, mass_high, 21)

    counts_opt, bins = np.histogram(diMuon_mass_BSC[mask_opt], bins=mass_bins, weights=total_weights[mask_opt])
    counts_nominal, bins = np.histogram(diMuon_mass_BSC[mask_nominal], bins=mass_bins, weights=total_weights[mask_nominal])
    counts_run2, bins = np.histogram(diMuon_mass_BSC[mask_run2], bins=mass_bins, weights=total_weights[mask_run2])
    counts_opt = counts_opt / counts_opt.sum()
    counts_nominal = counts_nominal / counts_nominal.sum()
    counts_run2 = counts_run2 / counts_run2.sum()

    hep.histplot(counts_opt, bins=bins, ax=axs[0], label=f"BDT > {bdt_low:.2f}")
    hep.histplot(counts_nominal, bins=bins, ax=axs[1], label=f"BDT > {bdt_low:.2f}")
    hep.histplot(counts_run2, bins=bins, ax=axs[2], label=f"BDT > {bdt_low:.2f}")

#ax.legend()
axs[2].legend(bbox_to_anchor=(1.02, 1), loc='upper left')
axs[0].set_title("Opt", y=-0.175)
axs[1].set_title("Nominal", y=-0.175)
axs[2].set_title("Run2", y=-0.175)
#plt.show()
#plt.savefig('../plots/sculpting_study/optunaBkgVBFscuplting.png', bbox_inches="tight")
#plt.savefig('../plots/sculpting_study/optunaBkgVBFscuplting.pdf', bbox_inches="tight")

In [ ]:
for i, var in enumerate(variables[:-3]):
    if i == 10 or i == 11:
        print(f'Removing {var[0]}')
    else:
        print(f'Keeping {var[0]}')
x_train_new = np.delete(x_train, [10,11], axis=1)
x_test_new = np.delete(x_test, [10,11], axis=1)
x_train_new

In [ ]:
y_train_opt_new = y_train.copy()
y_train_nominal_new = y_train.copy()
y_train_run2_new = y_train.copy()

y_test_opt_new = y_test.copy()
y_test_nominal_new = y_test.copy()
y_test_run2_new = y_test.copy()
print("finished copying")

opted_params = {'max_depth': 5, 'learning_rate': 0.17065960988185175, 'min_child_weight': 1.258974431208238,
                            'subsample': 0.9799252299722975, 'colsample_bytree': 0.9857090292326347, 
                            'reg_lambda': 0.17418443431820596, 'reg_alpha': 0.03970459451078197}

opt_model_new = xgb.XGBClassifier(
            **opted_params,
            n_estimators=100,
            verbosity=2,
            n_jobs=4,
            eval_metric=["logloss", "auc"],
            early_stopping_rounds=50,
            device='cuda',
        )

nominal_model_new = xgb.XGBClassifier(
            max_depth=3,
            learning_rate=0.1,
            n_estimators=400,
            verbosity=2,
            n_jobs=4,
            eval_metric=["logloss", "auc"],
            device='cuda',
        )

run2_model_new = xgb.XGBClassifier(
            max_depth = 4,
            n_estimators=1000,
            min_child_weight = 500000,
            learning_rate = 0.1,
            subsample = 0.5,
            verbosity=2,
            n_jobs=4,
            eval_metric=["logloss", "auc"],
            device='cuda',
        )

opt_model_new.fit(
            x_train_new,
            y_train_opt_new,
            sample_weight=w_train,
            eval_set=[(x_train_new, y_train_opt_new), (x_test_new, y_test_opt_new)],
            sample_weight_eval_set=[w_train, w_test],
        )

nominal_model_new.fit(
            x_train_new,
            y_train_nominal_new,
            sample_weight=w_train,
            eval_set=[(x_train_new, y_train_nominal_new), (x_test_new, y_test_nominal_new)],
            sample_weight_eval_set=[w_train, w_test],
        )

run2_model_new.fit(
            x_train_new,
            y_train_run2_new,
            sample_weight=w_train,
            eval_set=[(x_train_new, y_train_run2_new), (x_test_new, y_test_run2_new)],
            sample_weight_eval_set=[w_train, w_test],
        )

In [ ]:
y_pred_opt_test_new = opt_model_new.predict_proba(x_test_new)[:, 1]
y_pred_opt_train_new = opt_model_new.predict_proba(x_train_new)[:, 1]
x_total_new = np.concatenate((x_test_new,x_train_new))
y_opt_new = np.concatenate((y_pred_opt_test_new,y_pred_opt_train_new))

y_pred_nominal_test_new = nominal_model_new.predict_proba(x_test_new)[:, 1]
y_pred_nominal_train_new = nominal_model_new.predict_proba(x_train_new)[:, 1]
y_nominal_new = np.concatenate((y_pred_nominal_test_new,y_pred_nominal_train_new))

y_pred_run2_test_new = run2_model_new.predict_proba(x_test_new)[:, 1]
y_pred_run2_train_new = run2_model_new.predict_proba(x_train_new)[:, 1]
y_run2_new = np.concatenate((y_pred_nominal_test_new,y_pred_nominal_train_new))

In [ ]:
print(y_pred_opt_test_new)
y_pred_nominal_test_bkg_new = nominal_model_new.predict_proba(x_test_new[y_test == 0])[:, 1]
y_pred_nominal_test_sig_new = nominal_model_new.predict_proba(x_test_new[y_test == 1])[:, 1]
counts_bkg_test_new, bins = np.histogram(y_pred_nominal_test_bkg_new, bins=100, weights=x_test_p[y_test == 0][:, -1])
counts_sig_test_new, bins = np.histogram(y_pred_nominal_test_sig_new, bins=100, weights=x_test_p[y_test == 1][:, -1])

y_pred_nominal_train_bkg_new = nominal_model_new.predict_proba(x_train_new[y_train == 0])[:, 1]
y_pred_nominal_train_sig_new = nominal_model_new.predict_proba(x_train_new[y_train == 1])[:, 1]
counts_bkg_train_new, bins = np.histogram(y_pred_nominal_train_bkg_new, bins=100, weights=x_train_p[y_train == 0][:, -1])
counts_sig_train_new, bins = np.histogram(y_pred_nominal_train_sig_new, bins=100, weights=x_train_p[y_train == 1][:, -1])

y_pred_nominal_test_bkg = nominal_model.predict_proba(x_test[y_test == 0])[:, 1]
y_pred_nominal_test_sig = nominal_model.predict_proba(x_test[y_test == 1])[:, 1]
counts_bkg_test, bins = np.histogram(y_pred_nominal_test_bkg, bins=100, weights=x_test_p[y_test == 0][:, -1])
counts_sig_test, bins = np.histogram(y_pred_nominal_test_sig, bins=100, weights=x_test_p[y_test == 1][:, -1])

y_pred_nominal_train_bkg = nominal_model.predict_proba(x_train[y_train == 0])[:, 1]
y_pred_nominal_train_sig = nominal_model.predict_proba(x_train[y_train == 1])[:, 1]
counts_bkg_train, bins = np.histogram(y_pred_nominal_train_bkg, bins=100, weights=x_train_p[y_train == 0][:, -1])
counts_sig_train, bins = np.histogram(y_pred_nominal_train_sig, bins=100, weights=x_train_p[y_train == 1][:, -1])

fig, ax = plt.subplots()

hep.histplot(counts_bkg_test_new + counts_bkg_train_new, bins=bins, ax=ax, label="bkg - 2")
hep.histplot(counts_sig_test_new + counts_sig_train_new, bins=bins, ax=ax, label="sig - 2")
hep.histplot(counts_bkg_test + counts_bkg_train, bins=bins, ax=ax, label="bkg")
hep.histplot(counts_sig_test + counts_sig_train, bins=bins, ax=ax, label="sig")
#hep.histplot(counts_bkg_train_new, bins=bins, ax=ax, label="bkg train")
#hep.histplot(counts_sig_train_new, bins=bins, ax=ax, label="sig train")
ax.legend()

In [ ]:
bdt_divisions = 10
mass_low = 115
mass_high = 135
hep.style.use("CMS")

#fig, ax = plt.subplots()
fig, axs = plt.subplots(
    nrows=1,
    ncols=3,
    figsize=(18, 6),
    sharex=True,
)

hep.cms.label("", data=False, ax=axs[0]) 
axs[2].set_xlabel('$M_{\mu\mu}$')
axs[0].set_ylabel('Normalized Events')

bdt_score_opt = np.concatenate((y_pred_opt_test_new[y_test == 0],y_pred_opt_train_new[y_train == 0]))
bdt_score_nominal = np.concatenate((y_pred_nominal_test_new[y_test == 0],y_pred_nominal_train_new[y_train == 0]))
bdt_score_run2 = np.concatenate((y_pred_run2_test_new[y_test == 0],y_pred_run2_train_new[y_train == 0]))

for bdt_cut in np.linspace(0,0.9,bdt_divisions):
    bdt_spacing = 1/bdt_divisions
    bdt_low = bdt_cut
    bdt_high = bdt_low + bdt_spacing
    
    mask_opt = (bdt_score_opt > bdt_low) & (diMuon_mass_BSC > mass_low) & (diMuon_mass_BSC < mass_high)
    mask_nominal = (bdt_score_nominal > bdt_low) & (diMuon_mass_BSC > mass_low) & (diMuon_mass_BSC < mass_high)
    mask_run2 = (bdt_score_run2 > bdt_low) & (diMuon_mass_BSC > mass_low) & (diMuon_mass_BSC < mass_high)
    hep.style.use("CMS")

    mass_bins = np.linspace(mass_low, mass_high, 21)

    counts_opt, bins = np.histogram(diMuon_mass_BSC[mask_opt], bins=mass_bins, weights=total_weights[mask_opt])
    counts_nominal, bins = np.histogram(diMuon_mass_BSC[mask_nominal], bins=mass_bins, weights=total_weights[mask_nominal])
    counts_run2, bins = np.histogram(diMuon_mass_BSC[mask_run2], bins=mass_bins, weights=total_weights[mask_run2])
    counts_opt = counts_opt / counts_opt.sum()
    counts_nominal = counts_nominal / counts_nominal.sum()
    counts_run2 = counts_run2 / counts_run2.sum()

    hep.histplot(counts_opt, bins=bins, ax=axs[0], label=f"BDT > {bdt_low:.1f}")
    hep.histplot(counts_nominal, bins=bins, ax=axs[1], label=f"BDT > {bdt_low:.1f}")
    hep.histplot(counts_run2, bins=bins, ax=axs[2], label=f"BDT > {bdt_low:.1f}")

#ax.legend()
axs[2].legend(bbox_to_anchor=(1.02, 1), loc='upper left')
axs[0].set_title("Opt", y=-0.175)
axs[1].set_title("Nominal", y=-0.175)
axs[2].set_title("Run2", y=-0.175)
#plt.show()
#plt.savefig('../plots/sculpting_study/optunaBkgVBFscuplting.png', bbox_inches="tight")
#plt.savefig('../plots/sculpting_study/optunaBkgVBFscuplting.pdf', bbox_inches="tight")

In [ ]:
print("hi")

In [ ]:
from scipy.stats import ks_2samp
import cupy as cp

In [ ]:
(bkg_weight_sum / signal_weight_sum)

In [ ]:
def compute_significance(y_true, y_score, weights):

    thresholds = np.linspace(0, 1, 100)
    best_Z = 0

    sig_weights = weights[(y_true == 1)].copy() * (signal_weight_sum / bkg_weight_sum)
    bkg_weights = weights[(y_true == 0)].copy()
    total_sig = np.sum(sig_weights)
    total_bkg = np.sum(bkg_weights)
    base_Z = total_sig / np.sqrt(total_bkg)
    best_cut = 0
    for cut in thresholds:
        sig_passed = y_score[y_true == 1] > cut
        bkg_passed = y_score[y_true == 0] > cut

        S_high = np.sum(sig_weights[sig_passed])
        B_high = np.sum(bkg_weights[bkg_passed])

        S_low = np.sum(sig_weights[~sig_passed])
        B_low = np.sum(bkg_weights[~bkg_passed])

        if B_low > 0:
            Z_low = S_low / np.sqrt(B_low)
        else:
            Z_low = 0

        if B_high > 0:
            Z_high = S_high / np.sqrt(B_high)
        else:
            Z_high = 0

        Z = np.sqrt(Z_low**2 + Z_high**2)

        if Z > best_Z:
            best_Z = Z
            best_cut = cut
    print(f"Best Z: {best_Z} | base Z: {base_Z}")
    return best_Z, best_cut

In [ ]:
def compute_sculpting_penalty(dimuon_spectrum, y_score, BDT_cut):
    before = dimuon_spectrum
    after = dimuon_spectrum[y_score > BDT_cut]
    ks_stat, pval = ks_2samp(before, after)
    return ks_stat

In [ ]:
y_opt_bkg = np.concatenate((y_pred_opt_test[y_test == 0],y_pred_opt_train[y_train == 0]))
y_nominal_bkg = np.concatenate((y_pred_nominal_test[y_test == 0],y_pred_nominal_train[y_train == 0]))
y_run2_bkg = np.concatenate((y_pred_run2_test[y_test == 0],y_pred_run2_train[y_train == 0]))
opt_penalty = compute_sculpting_penalty(diMuon_mass_BSC, y_opt_bkg, 0.5)
nominal_penalty = compute_sculpting_penalty(diMuon_mass_BSC, y_nominal_bkg, 0.5)
run2_penalty = compute_sculpting_penalty(diMuon_mass_BSC, y_run2_bkg, 0.5)


y_opt_bkg_new = np.concatenate((y_pred_opt_test_new[y_test == 0],y_pred_opt_train_new[y_train == 0]))
y_nominal_bkg_new = np.concatenate((y_pred_nominal_test_new[y_test == 0],y_pred_nominal_train_new[y_train == 0]))
y_run2_bkg_new = np.concatenate((y_pred_run2_test_new[y_test == 0],y_pred_run2_train_new[y_train == 0]))
opt_penalty_new = compute_sculpting_penalty(diMuon_mass_BSC, y_opt_bkg_new, 0.5)
nominal_penalty_new = compute_sculpting_penalty(diMuon_mass_BSC, y_nominal_bkg_new, 0.5)
run2_penalty_new = compute_sculpting_penalty(diMuon_mass_BSC, y_run2_bkg_new, 0.5)

print("----------------------------------------")
print(f"Optimized penalty: {opt_penalty}")
print(f"Nominal penalty: {nominal_penalty}")
print(f"Run2 penalty: {run2_penalty}")
print("-------- No jet angles -----------------")
print(f"Optimized penalty: {opt_penalty_new}")
print(f"Nominal penalty: {nominal_penalty_new}")
print(f"Run2 penalty: {run2_penalty_new}")

In [ ]:
def objective(trial):
    y_train_copy = y_train.copy()
    y_test_copy = y_test.copy()

    params = {
        "max_depth": trial.suggest_int("max_depth", 2, 5),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "n_estimators": 1000,
        "device": "cuda",

        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 1000000.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),

        "reg_lambda": trial.suggest_float("reg_lambda", 1e-5, 10.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 1.0, log=True),

        "objective": "binary:logistic",
        "eval_metric": ["logloss", "auc"],
        "n_jobs": 4,
        "verbosity": 1,
        "early_stopping_rounds": 50,
    }

    model = xgb.XGBClassifier(**params)

    model.fit(
        x_train,
        y_train_copy,
        sample_weight=w_train,
        eval_set=[(x_train, y_train_copy), (x_test, y_test_copy)],
        sample_weight_eval_set=[w_train, w_test],
        verbose=False,
    )

    y_pred = model.predict_proba(x_test)[:, 1]
    
    Z, cut = compute_significance(y_test_copy, y_pred, sample_weights_test)

    diMuon_mass_BSC_test = x_test_p[:, -3][y_test == 0]
    penalty = compute_sculpting_penalty(diMuon_mass_BSC_test, y_pred[y_test_copy == 0], cut)
    print(f"Penalty: {penalty}")
    #if penalty > 0.02:
    #    print("This model is sculpting too far")
    #    Z = -999

    #auc = roc_auc_score(
    #    y_test,
    #    y_pred,
    #    sample_weight=w_test,
    #)
    coeff = 10
    return Z - coeff*penalty

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

print("Best Z:", study.best_value)
print("Best parameters:")
print(study.best_params)

In [ ]:
fig = vis.plot_optimization_history(study)
show(fig)

fig = vis.plot_param_importances(study)
show(fig)

fig = vis.plot_slice(
    study,
    params=[
        "max_depth",
        "learning_rate",
        "min_child_weight",
        "subsample",
        "colsample_bytree",
        "reg_lambda",
        "reg_alpha",
    ],
)
show(fig)

In [ ]:
best_params = study.best_params

final_model = xgb.XGBClassifier(
    **best_params,
    n_estimators=2000,
    objective="binary:logistic",
    eval_metric=["logloss","auc"],
    device='cuda',
    early_stopping_rounds=50,
    n_jobs=4,
    verbosity=1,
)
y_train_prime = y_train.copy()
y_test_prime = y_test.copy()
final_model.fit(
    x_train,
    y_train_prime,
    sample_weight=w_train,
    eval_set=[(x_test, y_test)],
    sample_weight_eval_set=[w_test],
    verbose=False,
)